In [6]:
"""
Wine Quality Regression - Optimal Pipeline
Phương pháp: XGBoost + Optuna HPO + Feature Engineering
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor


In [7]:
print("WINE QUALITY REGRESSION PIPELINE")

def load_and_encode(path, is_train=True):
    df = pd.read_csv(path)
    # Encode type: hỗ trợ cả dạng 0/1 lẫn "red"/"white"
    df["type"] = df["type"].str.strip(" ").map({"red": 1, "white": 0})
    return df

df = load_and_encode("data/train.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['quality'].value_counts().sort_index()}\n")



WINE QUALITY REGRESSION PIPELINE
Dataset shape: (6714, 13)
Target distribution:
quality
3      23
4     230
5    2365
6    2809
7    1062
8     212
9      13
Name: count, dtype: int64



In [8]:
def feature_engineering(df):
    df = df.copy()

    # Tỉ lệ hóa học quan trọng
    df["free_to_total_SO2"]    = df["free sulfur dioxide"] / (df["total sulfur dioxide"] + 1e-6)
    df["acidity_ratio"]        = df["fixed acidity"] / (df["volatile acidity"] + 1e-6)
    df["acid_sugar_ratio"]     = df["fixed acidity"] / (df["residual sugar"] + 1e-6)

    # Tương tác features
    df["alcohol_density"]      = df["alcohol"] * df["density"]
    df["sulphates_alcohol"]    = df["sulphates"] * df["alcohol"]
    df["sulphates_SO2"]        = df["sulphates"] * df["free sulfur dioxide"]

    # Log transform (giảm skewness)
    for col in ["residual sugar", "chlorides", "free sulfur dioxide",
                "total sulfur dioxide", "sulphates"]:
        df[f"log_{col.replace(' ', '_')}"] = np.log1p(df[col])

    # Polynomial features cho top predictors
    for col in ["alcohol", "volatile acidity", "sulphates"]:
        df[f"{col.replace(' ', '_')}_sq"] = df[col] ** 2

    return df

df_feat = feature_engineering(df)

X = df_feat.drop(columns=["quality"])
y = df_feat["quality"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]}")
print(f"Features sau engineering: {X.shape[1]}\n")

Train size: 5371 | Val size: 1343
Features sau engineering: 26



In [9]:
print("BASELINE COMPARISON")

baselines = {
    "CatBoost (default)":    CatBoostRegressor(iterations=300, random_state=42, verbose=0),
    "XGBoost (default)": xgb.XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "LightGBM (default)": lgb.LGBMRegressor(n_estimators=300, random_state=42, n_jobs=-1, verbose=-1),
}

for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse  = np.sqrt(mean_squared_error(y_val, preds))
    mae   = mean_absolute_error(y_val, preds)
    r2    = r2_score(y_val, preds)
    print(f"{name:<25} RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

BASELINE COMPARISON
CatBoost (default)        RMSE=0.6225  MAE=0.4768  R²=0.5057
XGBoost (default)         RMSE=0.5883  MAE=0.3965  R²=0.5584
LightGBM (default)        RMSE=0.5872  MAE=0.4302  R²=0.5601


In [11]:
print("OPTUNA HPO - XGBoost")

import optuna
OPTUNA_AVAILABLE = True

if OPTUNA_AVAILABLE:
    def objective(trial):
        params = {
            "n_estimators":       trial.suggest_int("n_estimators", 200, 1000),
            "max_depth":          trial.suggest_int("max_depth", 3, 10),
            "learning_rate":      trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":          trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree":   trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight":   trial.suggest_int("min_child_weight", 1, 10),
            "gamma":              trial.suggest_float("gamma", 0, 5),
            "reg_alpha":          trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
            "reg_lambda":         trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "eval_metric": "rmse",
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model = xgb.XGBRegressor(**params)
        scores = cross_val_score(model, X_train, y_train,
                                  cv=cv, scoring="neg_root_mean_squared_error")
        return -scores.mean()

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50, show_progress_bar=True)

    best_params = study.best_params
    best_params["random_state"] = 42
    best_params["n_jobs"] = -1
    print(f"\nBest params: {best_params}")
    print(f"Best CV RMSE: {study.best_value:.4f}")

else:
    # Fallback params (đã tune trước)
    best_params = {
        "n_estimators": 700,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "gamma": 0.1,
        "reg_alpha": 0.5,
        "reg_lambda": 1.0,
        "random_state": 42,
        "n_jobs": -1,
    }


OPTUNA HPO - XGBoost


[I 2026-06-30 07:54:40,245] A new study created in memory with name: no-name-8688965d-6a12-4532-8811-ecde05319e24
Best trial: 0. Best value: 0.688004:   2%|▏         | 1/50 [00:02<01:53,  2.31s/it]

[I 2026-06-30 07:54:42,555] Trial 0 finished with value: 0.6880044579505921 and parameters: {'n_estimators': 987, 'max_depth': 3, 'learning_rate': 0.012584635467761625, 'subsample': 0.6311725078214929, 'colsample_bytree': 0.6546094291832009, 'min_child_weight': 9, 'gamma': 1.0702541211446337, 'reg_alpha': 1.0523559319376522e-05, 'reg_lambda': 1.452868254590094e-07}. Best is trial 0 with value: 0.6880044579505921.


Best trial: 0. Best value: 0.688004:   4%|▍         | 2/50 [00:03<01:06,  1.40s/it]

[I 2026-06-30 07:54:43,311] Trial 1 finished with value: 0.694344162940979 and parameters: {'n_estimators': 352, 'max_depth': 7, 'learning_rate': 0.036385442240718185, 'subsample': 0.7884707359354262, 'colsample_bytree': 0.9193921637402176, 'min_child_weight': 1, 'gamma': 4.751643138374307, 'reg_alpha': 0.5088958150933224, 'reg_lambda': 6.664242183042897e-07}. Best is trial 0 with value: 0.6880044579505921.


...

[I 2026-06-30 07:57:33,029] Trial 47 finished with value: 0.6327426910400391 and parameters: {'n_estimators': 970, 'max_depth': 10, 'learning_rate': 0.03691913245695688, 'subsample': 0.6220042808603314, 'colsample_bytree': 0.962630065950842, 'min_child_weight': 5, 'gamma': 0.5856156654724376, 'reg_alpha': 0.12418497069332832, 'reg_lambda': 7.4324200299907066e-06}. Best is trial 42 with value: 0.6113558888435364.


Best trial: 42. Best value: 0.611356:  98%|█████████▊| 49/50 [02:57<00:03,  3.98s/it]

[I 2026-06-30 07:57:37,944] Trial 48 finished with value: 0.6265942931175232 and parameters: {'n_estimators': 440, 'max_depth': 8, 'learning_rate': 0.016458342931523835, 'subsample': 0.6939029598244387, 'colsample_bytree': 0.9987981358531509, 'min_child_weight': 2, 'gamma': 0.23569773532109542, 'reg_alpha': 0.029354882840793906, 'reg_lambda': 2.167722542576757e-08}. Best is trial 42 with value: 0.6113558888435364.


Best trial: 42. Best value: 0.611356: 100%|██████████| 50/50 [02:59<00:00,  3.59s/it]

[I 2026-06-30 07:57:39,747] Trial 49 finished with value: 0.6568650960922241 and parameters: {'n_estimators': 772, 'max_depth': 9, 'learning_rate': 0.15549794735589062, 'subsample': 0.6483709381571524, 'colsample_bytree': 0.880390786143758, 'min_child_weight': 6, 'gamma': 0.8491741249962743, 'reg_alpha': 0.00016507901264445343, 'reg_lambda': 0.008062508160309902}. Best is trial 42 with value: 0.6113558888435364.

Best params: {'n_estimators': 604, 'max_depth': 10, 'learning_rate': 0.020480960585500415, 'subsample': 0.6396295498071465, 'colsample_bytree': 0.9965409309409573, 'min_child_weight': 4, 'gamma': 0.013448608182392227, 'reg_alpha': 0.005233849991919126, 'reg_lambda': 4.3049595026289365e-06, 'random_state': 42, 'n_jobs': -1}
Best CV RMSE: 0.6114


In [12]:
print("FINAL MODEL - XGBoost Tuned")

xgb_tuned = xgb.XGBRegressor(**best_params)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

preds_xgb = xgb_tuned.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds_xgb))
mae  = mean_absolute_error(y_val, preds_xgb)
r2   = r2_score(y_val, preds_xgb)

print(f"XGBoost Tuned:  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")



FINAL MODEL - XGBoost Tuned
XGBoost Tuned:  RMSE=0.5581  MAE=0.3857  R²=0.6026


In [13]:
print("STACKING ENSEMBLE")

lgb_model = lgb.LGBMRegressor(
    n_estimators=700, learning_rate=0.03, max_depth=6,
    num_leaves=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42,
    n_jobs=-1, verbose=-1
)

cat_model = CatBoostRegressor(
    iterations=700, learning_rate=0.03, depth=6,
    l2_leaf_reg=1.0, random_state=42, verbose=0
)

stack = StackingRegressor(
    estimators=[
        ("xgb", xgb_tuned),
        ("lgb", lgb_model),
        ("cat",  cat_model),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    passthrough=False
)

stack.fit(X_train, y_train)
preds_stack = stack.predict(X_val)

rmse_s = np.sqrt(mean_squared_error(y_val, preds_stack))
mae_s  = mean_absolute_error(y_val, preds_stack)
r2_s   = r2_score(y_val, preds_stack)

print(f"Stacking:       RMSE={rmse_s:.4f}  MAE={mae_s:.4f}  R²={r2_s:.4f}")


STACKING ENSEMBLE
Stacking:       RMSE=0.5605  MAE=0.3911  R²=0.5992


In [14]:
print("TOP 15 FEATURE IMPORTANCES (XGBoost)")

importance = pd.Series(
    xgb_tuned.feature_importances_, index=X.columns
).sort_values(ascending=False)

for feat, score in importance.head(15).items():
    bar = "█" * int(score * 200)
    print(f"{feat:<35} {bar} {score:.4f}")

TOP 15 FEATURE IMPORTANCES (XGBoost)
alcohol_density                     ███████████████████████████████ 0.1564
alcohol_sq                          ██████████████████████ 0.1123
acidity_ratio                       ██████████ 0.0543
volatile acidity                    █████████ 0.0456
alcohol                             ████████ 0.0432
volatile_acidity_sq                 ████████ 0.0422
log_chlorides                       ███████ 0.0387
type                                ██████ 0.0348
sulphates_SO2                       ██████ 0.0336
acid_sugar_ratio                    ██████ 0.0317
log_total_sulfur_dioxide            ██████ 0.0315
free_to_total_SO2                   ██████ 0.0305
sulphates_alcohol                   █████ 0.0291
chlorides                           █████ 0.0286
pH                                  █████ 0.0284


In [15]:
print("KẾT QUẢ CUỐI CÙNG")
print(f"{'Model':<30} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 58)

results = {
    "CatBoost (default)":    baselines["CatBoost (default)"],
    "XGBoost (default)":    baselines["XGBoost (default)"],
    "LightGBM (default)":   baselines["LightGBM (default)"],
}

for name, model in results.items():
    p = model.predict(X_val)
    print(f"{name:<30} {np.sqrt(mean_squared_error(y_val, p)):>8.4f} "
          f"{mean_absolute_error(y_val, p):>8.4f} {r2_score(y_val, p):>8.4f}")

print(f"{'XGBoost Tuned (Optuna)':<30} {rmse:>8.4f} {mae:>8.4f} {r2:>8.4f}  ✓ Best single")
print(f"{'Stacking Ensemble':<30} {rmse_s:>8.4f} {mae_s:>8.4f} {r2_s:>8.4f}  ✓ Best overall")
print("=" * 60)

KẾT QUẢ CUỐI CÙNG
Model                              RMSE      MAE       R²
----------------------------------------------------------
CatBoost (default)               0.6225   0.4768   0.5057
XGBoost (default)                0.5883   0.3965   0.5584
LightGBM (default)               0.5872   0.4302   0.5601
XGBoost Tuned (Optuna)           0.5581   0.3857   0.6026  ✓ Best single
Stacking Ensemble                0.5605   0.3911   0.5992  ✓ Best overall


In [16]:
print("GENERATE SUBMISSION")

# Load test
df_test = load_and_encode("data/test.csv", is_train=False)
test_ids = df_test["id"]
df_test = df_test.drop(columns=["id"])

# Feature engineering (same as train)
df_test_feat = feature_engineering(df_test)

# Retrain stacking model trên TOÀN BỘ train data (không bỏ val)
X_full = df_feat.drop(columns=["quality"])
y_full = df_feat["quality"]

print("Retraining stacking model on full train data...")

xgb_final = xgb.XGBRegressor(**best_params)
lgb_final  = lgb.LGBMRegressor(
    n_estimators=700, learning_rate=0.03, max_depth=6,
    num_leaves=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42,
    n_jobs=-1, verbose=-1
)
cat_final = CatBoostRegressor(
    iterations=700, learning_rate=0.03, depth=6,
    l2_leaf_reg=1.0, random_state=42, verbose=0
)


stack_final = StackingRegressor(
    estimators=[
        ("xgb", xgb_final),
        ("lgb", lgb_final),
        ("rf",cat_final),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    passthrough=False
)
stack_final.fit(X_full, y_full)

# Predict
test_preds = stack_final.predict(df_test_feat)

# Clip về khoảng hợp lệ (Kaggle thường dùng giá trị nguyên hoặc float)
test_preds = np.clip(test_preds, 3, 9)

# Tạo submission
submission = pd.DataFrame({
    "id":      test_ids,
    "quality": test_preds
})
submission.to_csv("submission9.csv", index=False)

print(f"Saved submission.csv — {len(submission)} rows")
print(f"Prediction range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")
print(submission.head(10).to_string(index=False))

GENERATE SUBMISSION
Retraining stacking model on full train data...
Saved submission.csv — 820 rows
Prediction range: [3.495, 7.616]
  id  quality
1257 6.446156
6409 5.982959
 136 4.980711
1631 6.700305
6084 5.899677
5434 4.996298
1094 5.274778
5146 6.478415
5921 6.511049
1076 5.759628


In [17]:
#Save model
from joblib import dump, load
dump(model, 'my_model.joblib')
loaded_model = load('my_model.joblib')
